# Camada Silver: base_motivos

In [0]:
from pyspark.sql import functions as F, Window

catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
ACENTOS = "áàâãäéèêëíìîïóòôõöúùûüç"
SEM_ACENTO = "aaaaaeeeeiiiiooooouuuuc"


def chave(c):
    """Texto em minúsculas, sem acento, sem espaços nas pontas e com espaços internos únicos.
    Serve só para comparar grafias; o valor final vem do dicionário de mapeamento."""
    t = F.lower(F.trim(F.col(c).cast("string")))
    return F.regexp_replace(F.translate(t, ACENTOS, SEM_ACENTO), r"\s+", " ")


def mapear(c, mapa):
    """Troca o valor da coluna `c` pelo valor do dicionário. Valor fora do dicionário vira null
    (e as validações abaixo barram o notebook se isso acontecer)."""
    expr = None
    for k, v in mapa.items():
        expr = F.when(F.col(c) == k, v) if expr is None else expr.when(F.col(c) == k, v)
    return expr


def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")

## 1. Motivos

In [0]:
MAPA_CATEGORIA = {
    "cartao": "Cartão", "beneficios": "Benefícios", "financeiro": "Financeiro",
    "cadastral": "Cadastral", "atendimento": "Atendimento",
}
MAPA_CRITICIDADE = {"alta": "Alta", "media": "Média", "baixa": "Baixa"}

# Só para os motivos sem categoria na origem (chave = nome do motivo sem acento, em minúsculas)
IMPUTACAO_CATEGORIA = {
    "transferencia de agencia": "Cadastral",
    "contratacao de cartao adicional": "Cartão",
    "cancelamento de conta": "Financeiro",
}

b_motivos = spark.table(f"{bronze_schema}.base_motivos")
display(b_motivos)

In [0]:
motivos = ultima_versao(
    b_motivos.withColumn("id_motivo", F.trim(F.col("id_motivo")).cast("int")),
    ["id_motivo"],
)
motivos = (
    motivos
    .withColumn("nome_motivo", F.trim(F.col("nome_motivo")))
    .withColumn("k_nome", chave("nome_motivo"))
    .withColumn("k_cat", chave("categoria"))
    .withColumn("k_crit", chave("criticidade"))
    .withColumn("categoria_imputada", F.col("k_cat").isNull() | (F.col("k_cat") == ""))         # sem categoria na origem
    .withColumn("categoria", F.coalesce(
        mapear("k_cat", MAPA_CATEGORIA),
        F.when(F.col("categoria_imputada"), mapear("k_nome", IMPUTACAO_CATEGORIA)),
    ))
    .withColumn("criticidade", mapear("k_crit", MAPA_CRITICIDADE))
    .select("id_motivo", "nome_motivo", "categoria", "criticidade", "categoria_imputada")
    .orderBy("id_motivo")
)

checar("motivos: nenhum campo nulo", motivos.filter(
    F.col("id_motivo").isNull() | F.col("nome_motivo").isNull()
    | F.col("categoria").isNull() | F.col("criticidade").isNull()))
checar("motivos: id_motivo único", motivos.groupBy("id_motivo").count().filter("count > 1"))
checar("motivos: categoria é uma das 5 do case", motivos.filter(
    ~F.col("categoria").isin("Financeiro", "Cartão", "Cadastral", "Atendimento", "Benefícios")))
display(motivos)

## 2. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(motivos, "tb_base_motivos")